# Round 06 — Extreme PyTorch Performance Mock · GPU Workbench

**Integrated Resolution-style code review + OpenAI-style PyTorch optimization + Anthropic-style performance validation.** Original mock, not an official interview question.

This notebook is the launcher for the **complete 8,404-line repository**, not a replacement for the source files. Use the repository directly from Colab; edits to files remain in the Colab runtime unless you commit them or copy them to Drive.

**Before running any cells:** Runtime → Change runtime type → **T4 GPU / another available GPU**. Availability depends on your Colab quota.

Suggested sequence: (1) open `notebooks/00_blind_review.ipynb` and do 45 min blind review; (2) inspect repo code; (3) run correctness gates; (4) GPU baselines and profiler; (5) edit files and compare. Do **not** look for an answer key—none is supplied.

## 0 — Verify CUDA and installed versions

In [ ]:
import sys, platform, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Device memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print('No CUDA GPU: enable GPU in Runtime settings. CPU correctness tests still work.')

## 1 — Clone the full project (isolated working copy)

In [ ]:
import os, pathlib, subprocess
base = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
repo = base / 'round06_training_repo'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'interview/06-extreme-perf',
                    'https://github.com/jsjhoubo/Projects.git', str(repo)], check=True)
project = repo / 'round06_extreme_perf'
assert (project/'perfkit').is_dir() and (project/'tests').is_dir()
os.chdir(project)
print('Working directory:', pathlib.Path.cwd())
print('Python files:', len(list(project.rglob('*.py'))))

## 2 — Install only missing test dependencies

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytest') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest>=8'], check=True)
print('PyTorch supplied by runtime:', __import__('torch').__version__)
print('pytest:', __import__('pytest').__version__)

## 3 — Correctness gates (CPU-compatible; GPU not required)

This is the **existing baseline**, not proof that a future optimized implementation is correct. The full suite includes hundreds of parameterized checks.

In [ ]:
import subprocess, sys, time
start=time.perf_counter()
res=subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--disable-warnings', '--maxfail=1'],
                   cwd=project)
print('Tests exit code:', res.returncode, 'elapsed(s):', round(time.perf_counter()-start,1))
if res.returncode != 0:
    raise RuntimeError('Correctness gate failed; inspect the error before profiling')

## 4 — Inspect the modules without displaying solutions

In [ ]:
for rel in ['perfkit/model/attention.py','perfkit/model/decoder.py',
            'perfkit/train/engine.py','perfkit/data/batcher.py',
            'perfkit/serve/decoding.py','perfkit/infra/timing.py']:
    p=project/rel
    print(f'{rel:48s} {len(p.read_text().splitlines()):4d} lines')
print('Open any of these files through Colab’s Files pane to review/edit.')

## 5 — GPU training benchmark: fixed workloads

**Record baseline results first.** Run one workload at a time; start with smaller sizes on an unfamiliar GPU. Comparing the default research attention against the SDPA execution path is a diagnostic comparison, not itself a solution.

In [ ]:
import json
from benchmarks.bench_train import run as bench_train

device='cuda' if torch.cuda.is_available() else 'cpu'
workloads=[(4,32),(8,64),(16,128)] if device=='cuda' else [(2,16),(4,32)]
train_results=[]
for batch,seq in workloads:
    for backend in ['research','sdpa']:
        if device=='cuda':
            torch.cuda.empty_cache()
        row=bench_train(device,batch,seq,width=64,layers=1,steps=3,
                        diagnostics=True,backend=backend)
        train_results.append(row)
        print(json.dumps(row,indent=2,default=str))

## 6 — Attention microbenchmark

In [ ]:
import subprocess, sys
for backend in ['research','sdpa']:
    cmd=[sys.executable, '-m','benchmarks.bench_attention','--device',device,
         '--batch','4' if device=='cuda' else '2',
         '--seq','64' if device=='cuda' else '32','--backend',backend]
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd,cwd=project,check=True)

## 7 — PyTorch profiler and downloadable Chrome trace

Profiler adds overhead. **Do not** treat profiler time as the speedup score; use the independent benchmark above.

In [ ]:
from benchmarks.profile_train import profile_experiment
trace = project / 'gpu_profile_round06.json'
prof=profile_experiment(device,steps=2,trace=str(trace))
print('Trace saved to:',trace,trace.stat().st_size,'bytes')
# Optional: from google.colab import files; files.download(str(trace))

## 8 — Validate optimizations against a reference

After editing your optimized implementation, extend the existing tests to compare **logits, loss, gradients, optimizer updates, generated tokens**, and memory/latency under identical seeds. Floating-point backends can differ slightly; avoid requiring bit-for-bit equality for SDPA/AMP. Do not blindly treat baseline speed or outputs as correct without checking the mathematical contract.

In [ ]:
# Re-run after EVERY edit. This cell intentionally has no solution code.
res=subprocess.run([sys.executable,'-m','pytest','tests','-q','--disable-warnings','--maxfail=1'],cwd=project)
assert res.returncode == 0

## 9 — Preserve your work / save results

The Colab VM is ephemeral. Download changed source files or push a personal Git branch after testing. Do not overwrite the existing training branch. Save benchmark JSON and a profiler trace for before/after comparison.

In [ ]:
from pathlib import Path
out=project/'gpu_benchmark_results.json'
out.write_text(json.dumps(train_results,indent=2,default=str),encoding='utf-8')
print('Results:',out)
# In hosted Colab: from google.colab import files; files.download(str(out))

### Challenge reminders

- **45-minute blind reading:** can you identify the dominant bottleneck and explain the fix without writing code?
- **Hard mode:** make it faster in code, show runtime/memory evidence, and pass differential tests.
- **Extreme mode:** scaling, variable lengths, GPU synchronizations, attention/optimizer parity, checkpointing and decoding/KV-cache trade-offs.

For detailed problem descriptions see [`README.md`](https://github.com/jsjhoubo/Projects/blob/interview/06-extreme-perf/round06_extreme_perf/README.md).